# Extracción (Bronze Layer)

## Importacion de recursos
En esta celda se cargan las librerías base (Pandas, NumPy) y el módulo centralizado utils.py. La centralización de rutas e importaciones garantiza que el pipeline sea reproducible y que no existan discrepancias en las rutas de acceso a los datos entre los distintos colaboradores del proyecto.

In [1]:
import utils
import pandas as pd
import uuid

## Conexión entre Capas Raw y Bronze
Se establecen las conexiones para mover datos desde la capa Raw (datos crudos) hacia Bronze (datos extraídos). El uso de bases de datos independientes garantiza la integridad de la fuente original y facilita la trazabilidad en el pipeline de ingeniería.

In [2]:
conn_raw = utils.get_db_connection("raw")
conn_ext = utils.get_db_connection("extracted")

## Capa Bronze
Este bloque realiza la transición de los datos de la capa Raw a la capa Bronze. La justificación técnica de este paso es el enriquecimiento de los datos con metadatos de auditoría (UUID y Timestamps). Esto garantiza el linaje del dato, permitiendo rastrear cada registro individualmente y conocer su momento exacto de entrada al sistema.

In [3]:
for i in range(1, 6):
    df = pd.read_sql(f"SELECT * FROM raw_fuente_{i}", conn_raw)

    df['metadata_uuid'] = [str(uuid.uuid4()) for _ in range(len(df))]
    df['metadata_extraction_date'] = utils.get_timestamp()
    df['metadata_source'] = f"dispositivo_fuente_{i}"

    df.to_sql(f"extracted_fuente_{i}", conn_ext, if_exists="replace", index=False)
    print(f"✅ Metadatos agregados a fuente {i} (Capa Bronze).")

conn_raw.close()
conn_ext.close()

✅ Metadatos agregados a fuente 1 (Capa Bronze).
✅ Metadatos agregados a fuente 2 (Capa Bronze).
✅ Metadatos agregados a fuente 3 (Capa Bronze).
✅ Metadatos agregados a fuente 4 (Capa Bronze).
✅ Metadatos agregados a fuente 5 (Capa Bronze).


## Creacion de archivos JSON
Se genera el archivo cdc_control.json para implementar un control de captura de cambios (Change Data Capture). Este archivo es vital para que el pipeline sea eficiente, permitiendo identificar el estado de sincronización de las fuentes y asegurar que solo los datos nuevos o actualizados sean procesados en ejecuciones futuras.

In [4]:
extraction_stats = [{"fuente": f"fuente_{i}", "fecha": utils.get_timestamp()} for i in range(1, 6)]
cdc_data = {
    "ultima_extraccion": utils.get_timestamp(),
    "estado": "Sincronizado"
}

utils.save_log(utils.CDC_PATH, cdc_data)
print(f"✅ Control de CDC guardado en: {utils.CDC_PATH}")

✅ Control de CDC guardado en: G:\Mi unidad\Drive.alumno.udg.mx\Proyectos\Proyecto_Integrador\logs\cdc_control.json
